# Misinformation Detection Pipeline

### Changes in this version
- **Atomic claim extraction** replaces TF-IDF: spaCy NER filters sentences to only those containing verifiable named entities (people, orgs, locations, dates, quantities), then SVO parsing turns them into focused search queries. No more `return` and `fly` as topics.
- **API credential validation** runs before the build loop so a misconfigured key fails immediately with a clear message instead of silently failing 200 times.
- **Credibility database** (SQLite): each source domain gets a Beta-distribution credibility score updated by model predictions and user reports. Scores are blended with cross-source agreement when ranking evidence.
- **Structural node features** (no content): GCN sees evidence structure, not article content, so it generalizes across topics.

In [24]:
# Dependencies
%pip install spacy && python -m spacy download en_core_web_sm

import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    nltk.download('punkt', quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

ENCODER = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = 384

print('Setup complete.')


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------- ------------------------------ 2.9/12.8 MB 17.6 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 18.8 MB/s eta 0:00:01
     ---------------------------------- ---- 11.3/12.8 MB 19.3 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 17.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


c:\Users\bhada\anaconda3\Lib\threading.py:996: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  del self._target, self._args, self._kwargs
c:\Users\bhada\anaconda3\Lib\threading.py:996: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  del self._target, self._args, self._kwargs
c:\Users\bhada\anaconda3\Lib\threading.py:996: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  del self._target, self._args, self._kwargs
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bf484626-caca-440d-ba3a-c456b3345ffd)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0b681c05-f4d2-4ce0-b0b9-4984dff68713)')' thrown while requesting HEAD https://huggingface.co/sentence-transforme

Setup complete.


## Step 1: Fact Extraction & Knowledge Graph

In [25]:
import networkx as nx
from nltk.tokenize import sent_tokenize

def extract_facts(text: str) -> list[tuple[str, np.ndarray]]:
    """Split text into sentences and embed each one."""
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return []
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return list(zip(sentences, embeddings))


def build_knowledge_graph(facts: list[tuple[str, np.ndarray]], threshold: float = 0.75) -> nx.Graph:
    """Build graph where nodes are sentences, edges connect semantically similar ones."""
    G = nx.Graph()
    if not facts:
        return G
    sentences, embeddings = zip(*facts)
    embeddings = np.array(embeddings)
    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        G.add_node(i, text=sent, embedding=emb, source='input', weight=1.0)
    sim_matrix = cosine_similarity(embeddings)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(i, j, similarity=float(sim_matrix[i, j]))
    return G

## Step 2: Atomic Claim Extraction

**Why not TF-IDF?** TF-IDF scores word frequency, so it surfaces the most common tokens in the article — often generic verbs (`return`, `fly`, `expect`) or author-specific vocabulary rather than meaningful factual claims. Searching for `fly` returns nothing useful.

**The atomic claim approach:** A verifiable factual claim almost always involves a named entity — a person, organization, location, date, or quantity. spaCy's NER identifies these. We then filter sentences to only those containing at least one named entity, and extract the core subject-verb-object structure to form a focused, searchable query.

Example: `"President Biden signed the infrastructure bill on Monday"` → query: `"Biden signed infrastructure bill"`

This mirrors what FActScore does with an LLM, but without the API cost, by leaning on structured linguistic parsing instead.

In [26]:
# Entity types that indicate a verifiable factual claim.
# PERSON, ORG, GPE (geopolitical entity), DATE, CARDINAL (numbers) are the
# most reliable signals. We exclude ORDINAL, TIME, etc. as too vague.
VERIFIABLE_ENTITY_TYPES = {'PERSON', 'ORG', 'GPE', 'DATE', 'CARDINAL', 'NORP', 'FAC', 'PRODUCT', 'EVENT', 'LAW'}


def extract_svo(sent) -> str | None:
    """
    Extract a Subject-Verb-Object string from a spaCy sentence span.
    Returns None if no clear SVO structure is found.

    The resulting string is used as a search query, so it should be
    compact and contain only the most content-bearing tokens.
    """
    subject = None
    verb    = None
    obj     = None

    for token in sent:
        if token.dep_ in ('nsubj', 'nsubjpass') and subject is None:
            # Expand to include compound modifiers e.g. 'President Biden'
            subject = ' '.join(t.text for t in token.subtree
                               if t.dep_ in ('compound', 'amod', 'nsubj', 'nsubjpass')
                               or t == token)
        if token.pos_ == 'VERB' and token.dep_ in ('ROOT', 'relcl') and verb is None:
            verb = token.lemma_  # lemma so 'signed'/'signs'/'signing' all → 'sign'
        if token.dep_ in ('dobj', 'attr', 'pobj') and obj is None:
            obj = ' '.join(t.text for t in token.subtree
                           if t.dep_ in ('compound', 'amod', 'dobj', 'attr', 'pobj')
                           or t == token)

    parts = [p for p in [subject, verb, obj] if p]
    return ' '.join(parts) if len(parts) >= 2 else None


def extract_atomic_claims(text: str, max_claims: int = 3) -> list[str]:
    """
    Distill an article into atomic, searchable claim strings.

    Pipeline:
      1. Parse with spaCy to get sentences + named entities
      2. Keep only sentences with at least one verifiable entity type
      3. Score each qualifying sentence by number of entity types present
         (more entity types = more factually dense = better search query)
      4. Extract SVO structure from top-scored sentences
      5. Fall back to the full sentence if SVO extraction fails

    Returns at most max_claims strings, each suitable as a Google query.
    """
    doc = nlp(text[:5000])  # spaCy has a token limit; 5000 chars covers most articles

    scored_sentences = []
    for sent in doc.sents:
        sent_text = sent.text.strip()
        if len(sent_text) < 20:
            continue
        ent_types = {ent.label_ for ent in sent.ents}
        verifiable = ent_types & VERIFIABLE_ENTITY_TYPES
        if not verifiable:
            continue
        # Score = number of distinct verifiable entity types
        # (a sentence with PERSON + ORG + DATE is denser than one with just DATE)
        scored_sentences.append((len(verifiable), sent))

    # Sort by score descending, take top candidates
    scored_sentences.sort(key=lambda x: x[0], reverse=True)
    top_sents = [s for _, s in scored_sentences[:max_claims * 2]]  # extra pool for SVO failures

    claims = []
    for sent in top_sents:
        if len(claims) >= max_claims:
            break
        query = extract_svo(sent)
        if query is None:
            # SVO failed — fall back to full sentence, truncated to ~100 chars
            query = sent.text.strip()[:100]
        if query:
            claims.append(query)

    return claims


# Quick sanity check
sample = "President Biden signed the $1.2 trillion infrastructure bill on Monday. "\
         "The legislation passed the Senate 69-30. Critics argue the spending will fuel inflation."
print('Sample claims:', extract_atomic_claims(sample))

Sample claims: ['President Biden sign 1.2 infrastructure bill', 'legislation pass Senate']


## Step 3: Credibility Database

Each source domain gets a **Beta distribution** prior Beta(2, 2) — neutral credibility at 0.5 with weak initial confidence. Every model prediction and user report shifts the distribution. The score is just the Beta mean: `alpha / (alpha + beta)`.

Model updates are weighted at 0.3 (partial signal — the model could be wrong). User reports are weighted at 1.0 (treated as ground truth). An audit log records every signal so scores can be inspected or rolled back.

In [27]:
import sqlite3
import math
from urllib.parse import urlparse
from datetime import datetime

DB_PATH = 'source_credibility.db'
MODEL_WEIGHT = 0.3   # model predictions move scores slowly
USER_WEIGHT  = 1.0   # user reports are treated as ground truth


def get_db():
    return sqlite3.connect(DB_PATH)


def init_db():
    with get_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS sources (
                domain        TEXT PRIMARY KEY,
                alpha         REAL DEFAULT 2.0,
                beta          REAL DEFAULT 2.0,
                model_updates INTEGER DEFAULT 0,
                user_updates  INTEGER DEFAULT 0,
                last_updated  TEXT
            )
        ''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS credibility_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                domain      TEXT,
                signal_type TEXT,
                signal      TEXT,
                confidence  REAL,
                timestamp   TEXT
            )
        ''')
        conn.commit()

init_db()


def extract_domain(url: str) -> str:
    """'https://www.foxnews.com/article' → 'foxnews.com'"""
    return urlparse(url).netloc.replace('www.', '')


def get_credibility(domain: str) -> float:
    """Return Beta mean for domain. Unknown domains default to 0.5 (neutral)."""
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta FROM sources WHERE domain = ?', (domain,)
        ).fetchone()
    if row is None:
        return 0.5
    alpha, beta = row
    return alpha / (alpha + beta)


def get_credibility_report(domain: str) -> dict:
    """Full credibility summary including confidence interval and signal counts."""
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta, model_updates, user_updates FROM sources WHERE domain = ?',
            (domain,)
        ).fetchone()
    if row is None:
        return {'domain': domain, 'score': 0.5, 'n_signals': 0, 'confidence': 'none'}
    alpha, beta, model_updates, user_updates = row
    score    = alpha / (alpha + beta)
    variance = (alpha * beta) / ((alpha + beta) ** 2 * (alpha + beta + 1))
    n        = int(alpha + beta - 4)  # subtract prior mass
    return {
        'domain':        domain,
        'score':         round(score, 3),
        'std':           round(math.sqrt(variance), 3),
        'n_signals':     n,
        'model_updates': model_updates,
        'user_updates':  user_updates,
        'confidence':    'high' if n > 20 else 'medium' if n > 5 else 'low'
    }


def update_credibility(domain: str, signal: str, signal_type: str, confidence: float = 1.0):
    """
    Update Beta distribution for a domain.
    signal='real'  → increment alpha
    signal='fake'  → increment beta
    Weight is scaled by confidence so low-confidence model calls move scores less.
    """
    assert signal in ('real', 'fake')
    assert signal_type in ('model', 'user')
    weight = (MODEL_WEIGHT if signal_type == 'model' else USER_WEIGHT) * confidence
    now    = datetime.utcnow().isoformat()

    with get_db() as conn:
        conn.execute(
            'INSERT OR IGNORE INTO sources (domain, alpha, beta, last_updated) VALUES (?, 2.0, 2.0, ?)',
            (domain, now)
        )
        col = 'alpha' if signal == 'real' else 'beta'
        update_col = 'model_updates' if signal_type == 'model' else 'user_updates'
        conn.execute(
            f'UPDATE sources SET {col} = {col} + ?, {update_col} = {update_col} + 1, last_updated = ? WHERE domain = ?',
            (weight, now, domain)
        )
        conn.execute(
            'INSERT INTO credibility_log (domain, signal_type, signal, confidence, timestamp) VALUES (?,?,?,?,?)',
            (domain, signal_type, signal, confidence, now)
        )
        conn.commit()


def bulk_update_from_prediction(scored_docs: list[tuple[dict, float]],
                                 prediction: float,
                                 model_confidence: float):
    """
    After the GCN classifies an article, update all evidence source domains.
    Sources contributing to a fake article get their beta incremented;
    sources contributing to a real article get their alpha incremented.
    Weight = model_confidence × how much this source contributed (doc_score).
    """
    signal = 'fake' if prediction > 0.5 else 'real'
    for doc, doc_score in scored_docs:
        url = doc.get('link', '')
        if not url:
            continue
        domain = extract_domain(url)
        update_credibility(domain, signal, 'model',
                           confidence=model_confidence * doc_score)


print('Credibility DB initialised at', DB_PATH)

Credibility DB initialised at source_credibility.db


## Step 4: Google Search with Credential Validation & Disk Cache

In [28]:
%pip install duckduckgo_search

import time
import json
import os
import hashlib
from duckduckgo_search import DDGS

CACHE_DIR = 'search_cache'
os.makedirs(CACHE_DIR, exist_ok=True)


def _cache_path(query: str) -> str:
    return os.path.join(CACHE_DIR, hashlib.md5(query.encode()).hexdigest() + '.json')

def ddg_search(query: str, num_results: int = 10) -> list[dict]:
    """
    Search DuckDuckGo. Returns results in the same format the rest of
    the pipeline expects: list of dicts with 'link', 'title', 'snippet'.
    Results are cached to disk identically to before — same cache logic,
    same cache directory, re-running never re-fetches.
    """
    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'link':    r.get('href', ''),
                'title':   r.get('title', ''),
                'snippet': r.get('body', '')   # DDG calls it 'body', we normalize to 'snippet'
            })

    with open(cache_file, 'w') as f:
        json.dump(results, f)
    return results


def collect_evidence(text: str, k: int = 10) -> list[dict]:
    """
    No API key parameter needed anymore — DDG requires no credentials.
    Everything else is identical: atomic claims → search → deduplicate.
    """
    claims = extract_atomic_claims(text, max_claims=3)
    if not claims:
        return []

    seen_links  = set()
    all_results = []
    for claim in claims:
        try:
            results = ddg_search(claim, num_results=k)
        except Exception as e:
            print(f'  Search failed for "{claim[:60]}": {e}')
            continue
        for r in results:
            if r.get('link') not in seen_links:
                seen_links.add(r['link'])
                all_results.append(r)
        time.sleep(0.5)  # be nice to DDG and avoid hitting rate limits
    return all_results


Note: you may need to restart the kernel to use updated packages.


c:\Users\bhada\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  return process_handler(cmd, _system_body)
c:\Users\bhada\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  return process_handler(cmd, _system_body)
c:\Users\bhada\anaconda3\Lib\site-packages\IPython\utils\_process_win32.py:124: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  return process_handler(cmd, _system_body)


## Step 5: Cross-Source Agreement + Credibility Scoring

Two signals are blended to rank evidence documents:
1. **Cross-source agreement** (live): average cosine similarity to all other retrieved documents. High agreement → the claim is corroborated by many independent sources.
2. **Stored domain credibility** (from DB): Beta mean for the source's domain. Starts at 0.5 for unknown sources and shifts with experience.

`alpha=0.6` means we trust live consensus slightly more than history early on. As the DB grows, you can lower alpha.

In [29]:
def score_documents(documents: list[dict], alpha: float = 0.6) -> list[tuple[dict, float]]:
    """
    Rank documents by a blend of:
      - Cross-source agreement (live, unbiased)
      - Stored domain credibility (from the credibility DB)

    alpha: weight of cross-source agreement [0, 1]
    1-alpha: weight of stored credibility
    """
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []

    embeddings = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix = cosine_similarity(embeddings)
    n = len(documents)

    scored = []
    for i, doc in enumerate(documents):
        agreement   = (sim_matrix[i].sum() - 1.0) / max(n - 1, 1)
        domain      = extract_domain(doc.get('link', ''))
        credibility = get_credibility(domain)  # 0.5 for unknown domains
        combined    = alpha * float(agreement) + (1 - alpha) * credibility
        scored.append((doc, combined))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

## Step 6: Knowledge Graph Augmentation

In [30]:
def augment_knowledge_graph(G: nx.Graph,
                             scored_docs: list[tuple[dict, float]],
                             sim_threshold: float = 0.75,
                             score_threshold: float = 0.3) -> nx.Graph:
    """
    Add evidence nodes from ranked documents into the graph.
    Evidence node weights = document agreement+credibility score.
    Collects all changes first, then applies — safe from RuntimeError on G.nodes mutation.
    """
    existing_nodes = list(G.nodes(data=True))
    next_id = max(G.nodes()) + 1 if G.nodes() else 0
    new_nodes, new_edges = [], []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold:
            continue
        snippet = doc.get('snippet', '')
        if not snippet:
            continue
        evidence_facts = extract_facts(snippet)
        for fact_text, fact_emb in evidence_facts:
            nid = next_id
            next_id += 1
            new_nodes.append((nid, fact_text, fact_emb, doc_score))
            fact_emb_2d = fact_emb.reshape(1, -1)
            for existing_id, data in existing_nodes:
                sim = float(cosine_similarity(fact_emb_2d, data['embedding'].reshape(1, -1))[0, 0])
                if sim > sim_threshold:
                    new_edges.append((nid, existing_id, sim, doc_score))

    for nid, text, emb, score in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence', weight=score)
    for nid, eid, sim, score in new_edges:
        G.add_edge(nid, eid, similarity=sim, evidence_weight=score)
    return G

## Step 7: Structural Node Features & GCN

The GCN sees **only structural/relational features** — no sentence content. This forces it to learn *how well-supported a claim is* rather than *what the claim says*, so it generalises across topics.

In [31]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data

FEATURE_DIM = 8


def compute_node_features(G: nx.Graph) -> np.ndarray:
    """
    8 structural features per node — no content, pure evidence-relation structure.

      0: evidence_degree      — number of connected evidence nodes
      1: input_degree         — connections to other input (article) nodes
      2: avg_evidence_weight  — mean agreement+credibility score of evidence neighbours
      3: max_evidence_weight  — strongest single evidence connection
      4: avg_edge_similarity  — mean cosine sim across all edges
      5: evidence_ratio       — fraction of neighbours that are evidence nodes
      6: is_evidence          — 1 if from search results, 0 if from article
      7: source_weight        — document score for evidence nodes, 1.0 for input nodes
    """
    feature_list = []
    for node_id, data in G.nodes(data=True):
        neighbors         = list(G.neighbors(node_id))
        evidence_nbrs     = [n for n in neighbors if G.nodes[n].get('source') == 'evidence']
        input_nbrs        = [n for n in neighbors if G.nodes[n].get('source') == 'input']
        evidence_weights  = [G.edges[node_id, n].get('evidence_weight', 0.0) for n in evidence_nbrs]
        all_sims          = [G.edges[node_id, n].get('similarity', 0.0) for n in neighbors]

        feature_list.append([
            len(evidence_nbrs),
            len(input_nbrs),
            np.mean(evidence_weights) if evidence_weights else 0.0,
            np.max(evidence_weights)  if evidence_weights else 0.0,
            np.mean(all_sims)         if all_sims         else 0.0,
            len(evidence_nbrs) / max(len(neighbors), 1),
            1.0 if data.get('source') == 'evidence' else 0.0,
            data.get('weight', 1.0),
        ])

    arr = np.array(feature_list, dtype=np.float32)
    col_max = arr.max(axis=0)
    col_max[col_max == 0] = 1
    return arr / col_max


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    node_ids = list(G.nodes())
    id_map   = {nid: i for i, nid in enumerate(node_ids)}
    x        = torch.tensor(compute_node_features(G), dtype=torch.float)

    if G.edges():
        edges      = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)

    data = Data(x=x, edge_index=edge_index)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data


class FakeNewsGCN(torch.nn.Module):
    def __init__(self, input_dim: int = FEATURE_DIM, hidden_dim: int = 32, dropout: float = 0.5):
        super().__init__()
        self.conv1      = GCNConv(input_dim, hidden_dim)
        self.conv2      = GCNConv(hidden_dim, 16)
        self.classifier = torch.nn.Linear(16, 1)
        self.dropout    = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.classifier(x)  # raw logits


def train_gcn(train_data, val_data, epochs=100, lr=1e-3, patience=15):
    model     = FakeNewsGCN()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=16, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=16)

    best_val, no_improve, best_state = float('inf'), 0, None
    for epoch in range(epochs):
        model.train()
        t_loss = 0
        for batch in t_loader:
            optimizer.zero_grad()
            out  = model(batch.x, batch.edge_index, batch.batch).squeeze()
            loss = criterion(out, batch.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for batch in v_loader:
                out    = model(batch.x, batch.edge_index, batch.batch).squeeze()
                v_loss += criterion(out, batch.y.squeeze()).item()

        avg_t = t_loss / len(t_loader)
        avg_v = v_loss / len(v_loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}  train={avg_t:.4f}  val={avg_v:.4f}')

        if avg_v < best_val:
            best_val   = avg_v
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def evaluate_gcn(model, dataset):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    loader = DataLoader(dataset, batch_size=16)
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze())
            preds.extend(probs.tolist())
            labels.extend(batch.y.squeeze().tolist())
    binary = [1 if p > 0.5 else 0 for p in preds]
    return {
        'accuracy': accuracy_score(labels, binary),
        'f1':       f1_score(labels, binary),
        'auc':      roc_auc_score(labels, preds)
    }


def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f'Saved to {path}')

def load_model(path):
    m = FakeNewsGCN()
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

c:\Users\bhada\anaconda3\Lib\collections\__init__.py:449: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C6A3EE9A80>
  result = tuple_new(cls, iterable)


## Step 8: Dataset Build with Evidence Augmentation

After building each graph and running training, `bulk_update_from_prediction` updates the credibility DB for all evidence sources used. Over many articles the DB builds up a real signal about which domains consistently appear in misinformation vs reliable reporting.

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

df = pd.read_csv(os.path.abspath('data/fake_news_dataset.csv'))
df['label_binary'] = (df['label'] == 'fake').astype(int)
print(f"Dataset: {len(df)} articles | Balance: {df['label_binary'].value_counts().to_dict()}")


def build_dataset_with_evidence(df, text_col='text', label_col='label_binary', sample=None):
    if sample:
        df = df.sample(sample, random_state=42)

    pyg_data    = []
    all_scored  = []  # store scored_docs per article for credibility update after training
    n_augmented = 0
    n_fallback  = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text  = row[text_col]
        label = int(row[label_col])

        facts = extract_facts(text)
        if not facts:
            all_scored.append([])
            continue
        G = build_knowledge_graph(facts)

        scored_docs = []
        try:
            documents = collect_evidence(text, k=10)
            if documents:
                scored_docs = score_documents(documents)
                G           = augment_knowledge_graph(G, scored_docs)
                n_augmented += 1
            else:
                n_fallback += 1
        except Exception as e:
            print(f'  Evidence collection failed: {e}')
            n_fallback += 1

        pyg_data.append(graph_to_pyg(G, label=label))
        all_scored.append(scored_docs)

    print(f'\nBuilt {len(pyg_data)} graphs | Augmented: {n_augmented} | Fallback: {n_fallback}')
    return pyg_data, all_scored


pyg_dataset, all_scored_docs = build_dataset_with_evidence(
    df, sample=200
)

Dataset: 20000 articles | Balance: {1: 10056, 0: 9944}


Building graphs:   5%|▌         | 10/200 [00:10<03:23,  1.07s/it]c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C6A0159990>
  def __enter__(self):
c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C6A015A7A0>
  def __enter__(self):
c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C690386C50>
  def __enter__(self):
c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C6A3EE9A80>
  def __enter__(self):
c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C697567F10>
  def __enter__(self):
c:\Users\bhada\anaconda3\Lib\contextlib.py:136: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x000001C697567D30>
  de


Built 200 graphs | Augmented: 42 | Fallback: 158


## Step 9: Train, Evaluate, Update Credibility DB

In [37]:
from sklearn.model_selection import train_test_split

indices = list(range(len(pyg_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx,   test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gcn(train_data, val_data, epochs=100, patience=15)

metrics = evaluate_gcn(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy: {metrics["accuracy"]:.3f}')
print(f'  F1 Score: {metrics["f1"]:.3f}')
print(f'  AUC-ROC:  {metrics["auc"]:.3f}')

# Update credibility DB using model predictions on test set
# This is the feedback loop: every inference, regardless of whether it's
# training or production, gradually calibrates source credibility scores.
model.eval()
from torch_geometric.loader import DataLoader as PyGLoader
test_loader = PyGLoader(test_data, batch_size=1)  # batch_size=1 to align with all_scored_docs

n_db_updates = 0
with torch.no_grad():
    for i, (batch, scored_docs) in enumerate(zip(test_loader, [all_scored_docs[j] for j in test_idx])):
        logit      = model(batch.x, batch.edge_index, batch.batch).squeeze()
        prob       = torch.sigmoid(logit).item()
        confidence = abs(prob - 0.5) * 2  # 0 = uncertain, 1 = very confident
        if scored_docs and confidence > 0.3:  # only update DB if model is reasonably confident
            bulk_update_from_prediction(scored_docs, prob, confidence)
            n_db_updates += 1

print(f'\nCredibility DB updated for {n_db_updates} articles.')
save_model(model, 'fake_news_gcn.pt')

Split: 140 train | 30 val | 30 test


c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params

Epoch  10  train=0.6867  val=0.6930
Epoch  20  train=0.6900  val=0.6888
Epoch  30  train=0.6844  val=0.6891
Epoch  40  train=0.6867  val=0.6878
Epoch  50  train=0.6837  val=0.6876
Early stopping at epoch 57

Test Results:
  Accuracy: 0.633
  F1 Score: 0.476
  AUC-ROC:  0.594

Credibility DB updated for 0 articles.
Saved to fake_news_gcn.pt


In [34]:
# Inspect the credibility DB — see which domains have accumulated signal
with get_db() as conn:
    rows = conn.execute(
        'SELECT domain, alpha, beta, model_updates, user_updates '
        'FROM sources ORDER BY model_updates + user_updates DESC LIMIT 20'
    ).fetchall()

print(f'Top domains by signal count:')
print(f'{"Domain":<35} {"Score":>6} {"Signals":>8} {"Model":>7} {"User":>6}')
print('-' * 65)
for domain, alpha, beta, m_upd, u_upd in rows:
    score = alpha / (alpha + beta)
    n     = int(alpha + beta - 4)
    print(f'{domain:<35} {score:>6.3f} {n:>8} {m_upd:>7} {u_upd:>6}')

Top domains by signal count:
Domain                               Score  Signals   Model   User
-----------------------------------------------------------------
